## Step 1: Setup & Imports

In [ ]:
!pip install datasets --upgrade -q

import os
import csv
import re
import json
import glob
from datasets import load_dataset
from datetime import datetime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_FOLDER = "/content/drive/MyDrive/scam_detection/ScamContent-v5"
CHECKPOINT_FILE = os.path.join(OUTPUT_FOLDER, "_checkpoint.json")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f'Output: {OUTPUT_FOLDER}')
print(f'Checkpoint: {CHECKPOINT_FILE}')

## Step 2: Checkpoint Management System

In [ ]:
class CheckpointManager:
    """
    Manages checkpoint saving and loading for resumable processing.
    """

    def __init__(self, checkpoint_path):
        self.checkpoint_path = checkpoint_path
        self.checkpoint = None

    def load(self):
        """Load existing checkpoint or create new one."""
        if os.path.exists(self.checkpoint_path):
            try:
                with open(self.checkpoint_path, 'r') as f:
                    self.checkpoint = json.load(f)
                print('\n' + '='*70)
                print(' CHECKPOINT FOUND - RESUMING FROM LAST SESSION')
                print('='*70)
                print(f"Snapshot: {self.checkpoint.get('snapshot_id')}")
                print(f"Documents processed: {self.checkpoint.get('docs_processed', 0):,}")
                print(f"Messages extracted: {self.checkpoint.get('messages_extracted', 0):,}")
                print(f"Current batch: {self.checkpoint.get('current_batch', 1)}")
                print(f"Last updated: {self.checkpoint.get('last_updated')}")
                print('='*70 + '\n')
                return True
            except Exception as e:
                print(f'  Checkpoint corrupted: {e}')
                print('   Starting fresh...')
                self.checkpoint = None
                return False
        else:
            print('\n No checkpoint found. Starting fresh collection.\n')
            self.checkpoint = None
            return False

    def save(self, snapshot_id, docs_processed, messages_extracted, current_batch, buffer_size=0):
        """Save current progress."""
        checkpoint_data = {
            'snapshot_id': snapshot_id,
            'docs_processed': docs_processed,
            'messages_extracted': messages_extracted,
            'current_batch': current_batch,
            'buffer_size': buffer_size,
            'last_updated': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'status': 'in_progress'
        }

        try:
            with open(self.checkpoint_path, 'w') as f:
                json.dump(checkpoint_data, f, indent=2)
            self.checkpoint = checkpoint_data
        except Exception as e:
            print(f'Failed to save checkpoint: {e}')

    def get(self, key, default=None):
        """Get checkpoint value."""
        if self.checkpoint:
            return self.checkpoint.get(key, default)
        return default

    def mark_complete(self, snapshot_id):
        """Mark snapshot as fully processed."""
        if self.checkpoint:
            self.checkpoint['status'] = 'complete'
            self.checkpoint['completed_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

            try:
                with open(self.checkpoint_path, 'w') as f:
                    json.dump(self.checkpoint, f, indent=2)
                print('\nCheckpoint marked as COMPLETE')
            except Exception as e:
                print(f'Failed to mark complete: {e}')

    def delete(self):
        """Delete checkpoint file (start fresh)."""
        if os.path.exists(self.checkpoint_path):
            os.remove(self.checkpoint_path)
            print('Checkpoint deleted. Will start fresh.')
        self.checkpoint = None


# Initialize checkpoint manager
checkpoint_mgr = CheckpointManager(CHECKPOINT_FILE)

## Step 3: Filtering Functions (Same as Before)

In [ ]:
def is_first_person_narrative(text):
    """Detect first-person narratives."""
    text_lower = text.lower()

    narrative_patterns = [
        r'\bi\s+(received|got|was\s+sent)\s+(this|a|the)\s+(message|text|email|call)',
        r'\bmy\s+(phone|email|account)\s+(received|got|showed)',
        r'\bthey\s+(sent|texted|emailed|called)\s+me',
        r'\bsomeone\s+(sent|texted|emailed|called)\s+me',
        r'\bi\s+(clicked|replied|called|sent|gave)',
        r'\bthen\s+i\b', r'\bafter\s+i\b', r'\bwhen\s+i\b',
        r'\b(message|text|email)\s+(i|that\s+i)\s+(got|received)',
        r'\bthis\s+is\s+(what|the)\s+(message|text)',
    ]

    for pattern in narrative_patterns:
        if re.search(pattern, text_lower):
            return True

    story_markers = [
        'my story', 'my experience', 'what happened to me',
        'i should have', 'i thought it was', 'i believed',
        'looking back', 'in retrospect'
    ]

    if any(marker in text_lower for marker in story_markers):
        return True

    words = text_lower.split()
    if len(words) > 5:
        first_person = ['i', 'my', 'me', 'mine', 'myself']
        fp_count = sum(1 for word in words if word in first_person)
        if fp_count / len(words) > 0.08:
            return True

    return False


def is_meta_discussion(text):
    """Detect meta-discussion about scams."""
    text_lower = text.lower()

    meta_phrases = [
        'this is an example', 'here is an example', 'for example',
        'example of a scam', 'typical scam', 'common scam',
        'scam looks like', 'scam message might say',
        'you might receive', 'if you get a message',
        'watch out for', 'be aware of', 'warning signs',
        'how to spot', 'recognize these', 'these messages'
    ]

    if any(phrase in text_lower for phrase in meta_phrases):
        return True

    scammer_discussion = [
        'scammers use', 'scammers will', 'scammers often',
        'fraudsters', 'they try to', 'their goal is',
        'technique used by', 'tactic is to'
    ]

    if any(phrase in text_lower for phrase in scammer_discussion):
        return True

    return False


def is_scam_like(text):
    """Check for scam characteristics."""
    text_lower = text.lower()

    urgency = ['urgent', 'immediately', 'now', 'today', 'expires', 'limited', 'asap', 'hurry']
    rewards = ['won', 'prize', 'winner', 'congratulations', 'free', 'claim', '$', '£', '€']
    threats = ['suspended', 'blocked', 'terminated', 'legal', 'arrest', 'frozen', 'locked']
    actions = ['click', 'call', 'verify', 'confirm', 'reply', 'text', 'send', 'update']
    authority = ['irs', 'bank', 'government', 'police', 'security', 'account', 'amazon']

    score = 0
    score += sum(1 for word in urgency if word in text_lower)
    score += sum(1 for word in rewards if word in text_lower)
    score += sum(1 for word in threats if word in text_lower)
    score += sum(1 for word in actions if word in text_lower)
    score += sum(1 for word in authority if word in text_lower)

    if text.count('!') >= 2:
        score += 2
    if re.search(r'http[s]?://|www\.|bit\.ly|\d{3}[-.]?\d{3}[-.]?\d{4}', text_lower):
        score += 2
    if re.search(r'\b(code|ref|id|case|ticket)\s*[:=#]?\s*[A-Z0-9]{4,}', text, re.IGNORECASE):
        score += 1

    return score >= 6


def is_pure_scam_content(text):
    """Final validation: pure scam content only."""
    if not (20 <= len(text) <= 500):
        return False
    if not is_scam_like(text):
        return False
    if is_first_person_narrative(text):
        return False
    if is_meta_discussion(text):
        return False
    return True




## Step 4: Extraction Functions

In [ ]:
def extract_quoted_scams(text):
    scam_messages = []
    quotes = []
    quotes += re.findall(r'"([^"]{20,500})"', text)
    quotes += re.findall(r"'([^']{20,500})'", text)
    context_patterns = [
        r'(?:message|text|email)(?:\s+(?:said|read|was))?\s*[:"]\s*([^.!?"]{30,500}[.!?])',
    ]
    for pattern in context_patterns:
        quotes += re.findall(pattern, text, re.IGNORECASE)

    for quote in quotes:
        cleaned = quote.strip()
        if is_pure_scam_content(cleaned):
            scam_messages.append(cleaned)
    return scam_messages


def extract_short_scam_segments(text):
    segments = []
    sentences = re.split(r'[.!?]+', text)
    for sentence in sentences:
        cleaned = sentence.strip()
        if is_pure_scam_content(cleaned):
            segments.append(cleaned)
    return segments


def extract_from_scam_list(text):
    scam_messages = []
    numbered = re.findall(r'\d+\.\s*([^\n]{50,400})', text)
    bullets = re.findall(r'[-*•]\s*([^\n]{50,400})', text)
    examples = re.findall(r'(?:example|sample)\s*[:\-]\s*([^\n]{50,400})', text, re.IGNORECASE)
    all_candidates = numbered + bullets + examples
    for candidate in all_candidates:
        cleaned = candidate.strip()
        if is_pure_scam_content(cleaned):
            scam_messages.append(cleaned)
    return scam_messages


def extract_all_scam_content(text, url):
    scam_content = []
    scam_content.extend(extract_quoted_scams(text))
    scam_content.extend(extract_short_scam_segments(text))
    if 'scam' in url.lower() or 'fraud' in url.lower() or 'ftc' in url.lower():
        scam_content.extend(extract_from_scam_list(text))
    unique_scams = list(set(scam_content))
    final_filtered = [msg for msg in unique_scams if is_pure_scam_content(msg)]
    return final_filtered




## Step 5: Configuration

In [ ]:

SNAPSHOTS_TO_PROCESS = ['CC-MAIN-2024-10']
MAX_MESSAGES_PER_SNAPSHOT = 65000
BATCH_SIZE = 100
CHECKPOINT_INTERVAL = 100  # Save checkpoint every N documents
SKIP_MESSAGE_INTERVAL = 500000 # Print 'Skipping...' message every N documents

print(' Configuration:')
print(f'   Snapshots: {SNAPSHOTS_TO_PROCESS}')
print(f'   Max messages: {MAX_MESSAGES_PER_SNAPSHOT:,}')
print(f'   Batch size: {BATCH_SIZE}')
print(f'   Checkpoint every: {CHECKPOINT_INTERVAL} docs')
print(f'   Skip message every: {SKIP_MESSAGE_INTERVAL} docs')

## Step 6: Processing with Checkpoint Resume

In [ ]:
def process_snapshot_with_checkpoints(index_id, checkpoint_mgr):
    """
    Process snapshot with checkpoint resume capability.
    """
    print(f'\n{"="*70}')
    print(f'Processing: {index_id}')
    print(f'{"="*70}\n')

    # Load checkpoint
    has_checkpoint = checkpoint_mgr.load()

    # Determine starting point
    if has_checkpoint and checkpoint_mgr.get('snapshot_id') == index_id:
        # Resume from checkpoint
        docs_to_skip = checkpoint_mgr.get('docs_processed', 0)
        messages_extracted = checkpoint_mgr.get('messages_extracted', 0)
        file_batch = checkpoint_mgr.get('current_batch', 1)

        print(f' RESUMING from checkpoint:')
        print(f'   Will skip first {docs_to_skip:,} documents')
        print(f'   Already extracted: {messages_extracted:,} messages')
        print(f'   Next batch number: {file_batch}\n')
    else:
        # Start fresh
        docs_to_skip = 0
        messages_extracted = 0
        file_batch = 1
        print(' Starting fresh collection\n')

    try:
        # Load dataset
        print(' Loading dataset (this may take a minute)...')
        dataset = load_dataset(
            "HuggingFaceFW/fineweb",
            index_id,
            split="train",
            streaming=True
        )
        print('Dataset loaded!\n')

        buffer = []
        docs_processed = 0
        docs_this_session = 0

        for example in dataset:
            # Check if we've hit the limit
            if messages_extracted >= MAX_MESSAGES_PER_SNAPSHOT:
                print(f'\nReached target of {MAX_MESSAGES_PER_SNAPSHOT:,} messages')
                break

            docs_processed += 1

            # Skip already-processed documents
            if docs_processed <= docs_to_skip:
                if docs_processed % SKIP_MESSAGE_INTERVAL == 0:
                    print(f'Skipping... ({docs_processed:,}/{docs_to_skip:,})')
                continue

            docs_this_session += 1

            text = example.get('text', '')
            url = example.get('url', '')

            if len(text) < 200:
                continue

            # Extract scam content
            scam_messages = extract_all_scam_content(text, url)

            for msg in scam_messages:
                if messages_extracted < MAX_MESSAGES_PER_SNAPSHOT:
                    buffer.append([url, msg.replace('\n', ' ')])
                    messages_extracted += 1

            # Progress update
            if docs_this_session % 500 == 0:
                print(f' Session docs: {docs_this_session:,} | '
                      f'Total docs: {docs_processed:,} | '
                      f'Extracted: {messages_extracted:,}')

            # Save checkpoint periodically
            if docs_this_session % CHECKPOINT_INTERVAL == 0:
                checkpoint_mgr.save(
                    snapshot_id=index_id,
                    docs_processed=docs_processed,
                    messages_extracted=messages_extracted,
                    current_batch=file_batch,
                    buffer_size=len(buffer)
                )
                print(f' Checkpoint saved (docs: {docs_processed:,}, messages: {messages_extracted:,})')

            # Write batch
            if len(buffer) >= BATCH_SIZE:
                csv_path = os.path.join(
                    OUTPUT_FOLDER,
                    f'{index_id}_scam_batch{file_batch}.csv'
                )

                with open(csv_path, 'w', newline='', encoding='utf-8') as f:
                    writer = csv.writer(f)
                    writer.writerow(['url', 'text'])
                    writer.writerows(buffer[:BATCH_SIZE])

                print(f' Batch {file_batch}: {BATCH_SIZE} messages → {os.path.basename(csv_path)}')
                file_batch += 1
                buffer = buffer[BATCH_SIZE:]

                # Save checkpoint after each batch
                checkpoint_mgr.save(
                    snapshot_id=index_id,
                    docs_processed=docs_processed,
                    messages_extracted=messages_extracted,
                    current_batch=file_batch,
                    buffer_size=len(buffer)
                )

        # Write final batch
        if buffer:
            csv_path = os.path.join(
                OUTPUT_FOLDER,
                f'{index_id}_scam_batch{file_batch}.csv'
            )
            with open(csv_path, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(['url', 'text'])
                writer.writerows(buffer)
            print(f' Final batch: {len(buffer)} messages → {os.path.basename(csv_path)}')

        # Mark complete
        checkpoint_mgr.mark_complete(index_id)

        # Summary
        print(f'\n{"="*70}')
        print(f'SNAPSHOT COMPLETE: {index_id}')
        print(f'{"="*70}')
        print(f'Total documents processed: {docs_processed:,}')
        print(f'Documents this session: {docs_this_session:,}')
        print(f'Messages extracted: {messages_extracted:,}')
        print(f'Batches created: {file_batch}')
        print(f'{"="*70}\n')

    except KeyboardInterrupt:
        print('\nInterrupted by user')
        print(f'Checkpoint saved. Run again to resume from doc {docs_processed:,}')
        checkpoint_mgr.save(
            snapshot_id=index_id,
            docs_processed=docs_processed,
            messages_extracted=messages_extracted,
            current_batch=file_batch,
            buffer_size=len(buffer)
        )
    except Exception as e:
        print(f'\nError: {e}')
        print(f'Checkpoint saved. Run again to resume from doc {docs_processed:,}')
        checkpoint_mgr.save(
            snapshot_id=index_id,
            docs_processed=docs_processed,
            messages_extracted=messages_extracted,
            current_batch=file_batch,
            buffer_size=len(buffer)
        )
        import traceback
        print(traceback.format_exc())

## Step 7: RUN EXTRACTION (With Auto-Resume)

In [ ]:
# Process all snapshots
for snapshot in SNAPSHOTS_TO_PROCESS:
    process_snapshot_with_checkpoints(snapshot, checkpoint_mgr)

print('\n🎉 ALL SNAPSHOTS PROCESSED!')
print(f'📁 Output: {OUTPUT_FOLDER}')

## Step 8: Manual Controls (Optional)

In [ ]:
# View current checkpoint status
checkpoint_mgr.load()